# **Phase 2 - Transfer Learning & Data Augmentations**

Please follow the notebook in order. Make sure to add code where indicated by `''' TODO '''` or `# YOUR CODE HERE`. Please answer all questions indicated with '❓', and answer where it says 'ANSWER HERE'.

Ensure your notebook is easy to follow. All written report answers should be provided in the notebook itself.

In [1]:
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.datasets as datasets
import torchvision.transforms as transforms
import time
import os
import copy

import matplotlib.pyplot as plt
import numpy as np
import torchvision

## **Overview**
1. CIFAR10 Dataset
2. Data Augmentations
3. Transfer learning with ResNet Architecture
4. Training ResNet Architecture

# **1. CIFAR10 Dataset**

This is an RGB dataset with 10 classes:

*plane, car, bird, cat, deer, dog, frog, horse, ship, truck*


## **1.1 Download and Load CIFAR10**

You can find CIFAR10 in ```torchvision.datasets.CIFAR10()``` or see [here](https://pytorch.org/vision/0.15/generated/torchvision.datasets.CIFAR10.html#torchvision.datasets.CIFAR10). Please Download the dataset and use Dataloaders to load it. This is identical to loading MNIST.

In [2]:
transform = transforms.Compose(
    [transforms.ToTensor(),
     transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])

batch_size = 64

# Load CIFAR-10 training set
train_set = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)

train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)

# Load CIFAR-10 test set
testset = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)

test_loader = DataLoader(testset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

classes = ('plane', 'car', 'bird', 'cat',
           'deer', 'dog', 'frog', 'horse', 'ship', 'truck')

100.0%


NameError: name 'DataLoader' is not defined

## **1.2 Display CIFAR10 Images**

Similar to Phase 1, please complete `show_imgs()` function to display 10 images. However, this time we want to display exactly one image per class. There are 10 classes total. Show a random image per class with their label.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def unnormalize(img):
    """Undo Normalize((0.5,)*3, (0.5,)*3) and convert CxHxW tensor -> HxWxC numpy."""
    img = img * 0.5 + 0.5
    return np.clip(img.permute(1, 2, 0).numpy(), 0, 1)

def show_imgs(dataloader, classes):
    """Shows exactly one random image per class (10 images), each titled with its label."""
    found = {}
    for images, labels in dataloader:          # loader is shuffled -> random picks
        for img, lbl in zip(images, labels):
            lbl = lbl.item()
            if lbl not in found:
                found[lbl] = img
        if len(found) == len(classes):
            break

    fig, axes = plt.subplots(1, len(classes), figsize=(2 * len(classes), 2.5))
    for c, ax in enumerate(axes):
        ax.imshow(unnormalize(found[c]))
        ax.set_title(classes[c])
        ax.axis('off')
    plt.tight_layout()
    plt.show()

In [ ]:
show_imgs(train_loader, classes)

# **2. Data Augmentations**

## **2.1 Use Built-in Augmentations**

Use augmentations from the `torchvision.transforms` library [
link](https://pytorch.org/vision/0.9/transforms.html). Apply the following. For the parameters, please test and choose suitable ones.


*   Horizontal Flip
*   Random Crop
*   Color Jitter
*   Random Resized Crop

Try each of these individually and display the augmented images using your `show_imgs()` function.

In [ ]:
# Horizontal Flip  (p=1.0 just for the demo so every image is visibly flipped; we train with p=0.5)
aug_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=1.0),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# Define a dataset & loader with augmentations
train_aug_loader = DataLoader(datasets.CIFAR10(root='./data', train=True, download=True, transform=aug_transform),
                              batch_size=batch_size, shuffle=True)

# Display images with augmentations
show_imgs(train_aug_loader, classes)

In [ ]:
# Random Crop: pad 4 px (reflect) then crop back to 32x32 -> random shifts of up to 4 px
aug_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4, padding_mode='reflect'),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# Define a dataset & loader with augmentations
train_aug_loader = DataLoader(datasets.CIFAR10(root='./data', train=True, download=True, transform=aug_transform),
                              batch_size=batch_size, shuffle=True)

# Display images with augmentations
show_imgs(train_aug_loader, classes)

In [ ]:
# Color Jitter
aug_transform = transforms.Compose([
    transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# Define a dataset & loader with augmentations
train_aug_loader = DataLoader(datasets.CIFAR10(root='./data', train=True, download=True, transform=aug_transform),
                              batch_size=batch_size, shuffle=True)

# Display images with augmentations
show_imgs(train_aug_loader, classes)

In [ ]:
# Random Resized Crop: crop 60-100% of the image area with a random aspect ratio, resize back to 32x32
aug_transform = transforms.Compose([
    transforms.RandomResizedCrop(32, scale=(0.6, 1.0), ratio=(3 / 4, 4 / 3)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# Define a dataset & loader with augmentations
train_aug_loader = DataLoader(datasets.CIFAR10(root='./data', train=True, download=True, transform=aug_transform),
                              batch_size=batch_size, shuffle=True)

# Display images with augmentations
show_imgs(train_aug_loader, classes)

### ❓ **Q 2.1.1 Please explain what Data augmentations are and their use. What is the purpose of making them random?**

**Answer:** Data augmentation means applying label-preserving transformations (flips, crops, colour changes, rotations, etc.) to the training images so the model sees many plausible variations of each example. Its uses:

- **More effective training data:** it enlarges and diversifies the dataset without collecting new images.
- **Regularisation:** the model can't simply memorise exact pixel patterns, which reduces overfitting and improves generalisation to the test set.
- **Invariance:** it teaches the model that the class doesn't change under things like position, orientation, lighting or scale. These are the kinds of variation real test images contain.

**Why random:** augmentations are sampled randomly *every time an image is loaded*, so each epoch the network sees a different version of the same image. If the transform were fixed (e.g. always flipped), we would just have a different fixed dataset that the model could memorise just as easily, and it would learn nothing about invariance. Randomness (random parameters and random on/off with a probability) creates an effectively unlimited stream of new samples drawn from the distribution of plausible images. Augmentation is only applied to **training** data; the test set stays clean so that evaluation is consistent.

___
### ❓ **Q 2.1.2 Please explain all of the augmentations you are using. What is the value of each of them?**

**Answer:**

- **Random Horizontal Flip (p = 0.5):** mirrors the image left–right. A plane, car or cat facing left is the same class as one facing right, so this effectively doubles the data and teaches left/right invariance. (We avoid vertical flips: upside-down cars and animals are unrealistic.)
- **Random Crop (32, padding = 4):** pads the image by 4 pixels and takes a random 32×32 crop, which shifts the object by up to ±4 px. This teaches **translation invariance**, so the model doesn't rely on the object being perfectly centred. It is the standard CIFAR augmentation.
- **Color Jitter (brightness/contrast/saturation 0.4, hue 0.1):** randomly changes lighting and colour, which simulates different cameras, times of day and weather. The model learns to rely on shape and texture rather than exact colour (e.g. a red car versus a blue car). Hue is kept small so colours don't become completely unnatural.
- **Random Resized Crop (scale 0.6–1.0, aspect 3/4–4/3):** crops a random region of 60–100% of the image with a random aspect ratio and resizes it back to 32×32. This adds **scale and aspect-ratio invariance** (objects appear closer, farther or slightly stretched) and forces the model to recognise objects from partial views. The scale is kept ≥ 0.6 because 32×32 images are tiny, and very small crops would often cut out the object entirely.

## **2.2 Custom Augmentations**

Create a custom data augmentation function. Please do rotation, randomly between -30 and 30 degrees. You must code this yourself, do not use the torchvision function.

[This tutorial ](https://pytorch.org/tutorials/beginner/data_loading_tutorial.html) may be useful. You can use `PIL` image manupilation libraries, but NOT torchvision ones. Do not import any other libraries. You should only need:



```
from PIL import Image
import random
```





Apply this custom augmentation to the images and display the results using `show_imgs()`.

In [ ]:
from PIL import Image
import random

class RandomRotate:
    """Rotate a PIL image by a random angle in [min_angle, max_angle] degrees (PIL only, no torchvision)."""
    def __init__(self, min_angle=-30, max_angle=30, fill=(0, 0, 0)):
        self.min_angle = min_angle
        self.max_angle = max_angle
        self.fill = fill

    def __call__(self, img):
        angle = random.uniform(self.min_angle, self.max_angle)
        # expand=False keeps the 32x32 size; bilinear resampling for smooth edges
        return img.rotate(angle, resample=Image.BILINEAR, expand=False, fillcolor=self.fill)

    def __repr__(self):
        return f'{self.__class__.__name__}(min_angle={self.min_angle}, max_angle={self.max_angle})'

# Apply Transform
rotate_transform = transforms.Compose([
    RandomRotate(-30, 30),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

train_loader_rotate = DataLoader(datasets.CIFAR10(root='./data', train=True, download=True, transform=rotate_transform),
                                 batch_size=batch_size, shuffle=True)

# Display images with the custom augmentation
show_imgs(train_loader_rotate, classes)

### ❓ **Q 2.2.1 Explain how you would determine what is a good lower and upper limit for rotation? We use -30 to 30, why does that make sense?**

**Answer:** The range should cover the orientations the model will realistically meet at test time, while never changing the label or destroying the content. To choose it:

1. **Think about the domain:** how tilted can objects in real photos be? CIFAR images are photos taken roughly upright, so planes banking, a tilted camera or an animal turning its head give modest rotations, but almost never upside-down objects.
2. **Visualise:** apply increasing ranges (±10°, ±30°, ±60°, ±90°) with `show_imgs()` and check that the images still look natural and correctly labelled. Large rotations on 32×32 images also introduce big black corners and cut off parts of the object.
3. **Validate empirically:** treat the range as a hyperparameter and choose the value that gives the best validation accuracy. Too small adds little regularisation; too large adds unrealistic samples and lowers accuracy.

**Why −30° to 30° makes sense:** it covers natural tilt (camera roll, banking planes, animals at an angle) and is symmetric, because tilting left is as likely as tilting right. It is also small enough that objects stay recognisable, the label is unchanged, and the black fill-in at the corners stays small. Larger angles would create orientations that don't appear in the data (e.g. a 90° car) and could even change the meaning for orientation-sensitive classes, as with 6 vs 9 in digits.

____

# **3. Transfer Learning**
Load ResNet18 model from ```torchvision.models.resnet18()```, make sure to load pretrained model with [weights](https://pytorch.org/vision/stable/models.html).
You will need to modify the final layer. ResNet outputs 100 labels - please change to the correct output dimension.

In [ ]:
# Note: ImageNet-pretrained ResNet18 actually outputs 1000 classes; we replace its final fc layer with a
# 10-way layer for CIFAR10. Helper that builds a fresh pretrained model (used for every experiment below
# so each run starts from the same ImageNet weights).
from torchvision.models import resnet18, ResNet18_Weights

def make_resnet18(num_classes=10):
    model = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
    model.fc = nn.Linear(model.fc.in_features, num_classes)   # 512 -> 10
    return model

def count_trainable(model):
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f'Trainable params: {trainable:,} / {total:,} ({100 * trainable / total:.1f}%)')

## **3.1 ResNet18**

In [ ]:
from torchvision.models import resnet18, ResNet18_Weights

# Instantiate Pretrained ResNet18
resnet18_model = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)

# Modify final layer (512 -> 1000 ImageNet classes  ==>  512 -> 10 CIFAR10 classes)
resnet18_model.fc = nn.Linear(resnet18_model.fc.in_features, len(classes))

# Load onto GPU
resnet18_model = resnet18_model.to(device)

# View Model to validate
print(resnet18_model)

## **3.2 Train ResNet on CIFAR10**

### **3.2.1 Train and Test Functions**
You can refer to Phase_1.ipynb to complete these functions. In ```train()```, rather than saving the accuracy and loss per iteration, you will need to save the loss and accuracy ***per epoch***. Plot loss and accuracy plots for both train and validation sets.

In [ ]:
# Train Function
def train(model, loss_fn, optimizer, train_loader, test_loader, num_epochs, device, scheduler=None):
    """Trains the model, recording loss & accuracy PER EPOCH for train and validation (test) sets.
       Keeps the weights from the best validation epoch."""
    model.to(device)
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': [], 'epoch_time': []}
    best_acc, best_wts = 0.0, copy.deepcopy(model.state_dict())

    for epoch in range(num_epochs):
        start = time.time()
        model.train()
        running_loss, correct, total = 0.0, 0, 0
        for images, labels in train_loader:
            images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
            optimizer.zero_grad()
            outputs = model(images)
            loss = loss_fn(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * labels.size(0)
            correct += (outputs.argmax(1) == labels).sum().item()
            total += labels.size(0)
        if scheduler is not None:
            scheduler.step()

        train_loss, train_acc = running_loss / total, 100 * correct / total
        val_loss, val_acc = evaluate(model, loss_fn, test_loader, device)
        elapsed = time.time() - start
        for k, v in zip(history.keys(), [train_loss, train_acc, val_loss, val_acc, elapsed]):
            history[k].append(v)
        if val_acc > best_acc:
            best_acc, best_wts = val_acc, copy.deepcopy(model.state_dict())

        print(f'Epoch [{epoch + 1}/{num_epochs}] Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}% | '
              f'Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}% | Time: {elapsed:.1f}s')

    print(f'Best val acc: {best_acc:.2f}% | Total time: {sum(history["epoch_time"]):.1f}s')
    model.load_state_dict(best_wts)
    return history


def evaluate(model, loss_fn, loader, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            total_loss += loss_fn(outputs, labels).item() * labels.size(0)
            correct += (outputs.argmax(1) == labels).sum().item()
            total += labels.size(0)
    return total_loss / total, 100 * correct / total


# Test Function
def test_accuracy(model, test_loader, device):
    """Prints overall + per-class accuracy; returns (accuracy, all_preds, all_labels)."""
    model.to(device).eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in test_loader:
            outputs = model(images.to(device))
            all_preds.append(outputs.argmax(1).cpu())
            all_labels.append(labels)
    all_preds, all_labels = torch.cat(all_preds).numpy(), torch.cat(all_labels).numpy()
    acc = 100 * (all_preds == all_labels).mean()
    print(f'Test Accuracy: {acc:.2f}%')
    for i, c in enumerate(classes):
        mask = all_labels == i
        print(f'  {c:>6}: {100 * (all_preds[mask] == i).mean():.2f}%')
    return acc, all_preds, all_labels


# Plot Learning Curves
def plot_learning_curve(history, title=''):
    epochs = range(1, len(history['train_loss']) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
    axes[0].plot(epochs, history['train_loss'], 'o-', label='Train')
    axes[0].plot(epochs, history['val_loss'], 'o-', label='Validation')
    axes[0].set_title(f'{title} Loss'); axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
    axes[1].plot(epochs, history['train_acc'], 'o-', label='Train')
    axes[1].plot(epochs, history['val_acc'], 'o-', label='Validation')
    axes[1].set_title(f'{title} Accuracy'); axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy (%)')
    for ax in axes:
        ax.legend(); ax.grid(alpha=.3)
    plt.tight_layout()
    plt.show()

### **3.2.2 Train and Plot Learning Curves**

Retrain the ResNet18 on CIFAR10 Dataset. You can chose the optimizer and loss function. Train first with then without data augmentation to compare. When using data augmentations, you can choose which augmentations to use!

In [ ]:
## TRAINING WITHOUT AUGMENTATIONS

# Define Parameters
num_epochs = 10
lr = 3e-4          # small lr: we are fine-tuning pretrained weights, so we don't want to wreck them
torch.manual_seed(0)
resnet18_noaug = make_resnet18().to(device)

# Define Loss func and Optimizer
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(resnet18_noaug.parameters(), lr=lr)

# Train Model
hist_noaug = train(resnet18_noaug, loss_fn, optimizer, train_loader, test_loader, num_epochs, device)

# Plot Learning Curves
plot_learning_curve(hist_noaug, 'ResNet18 (no augmentation)')

# Evaluate on Test Set
acc_noaug, preds_noaug, labels_noaug = test_accuracy(resnet18_noaug, test_loader, device)

In [ ]:
## TRAINING WITH AUGMENTATIONS

# Augmentations chosen: random crop (shift), horizontal flip and mild colour jitter.
# (Rotation and strong resized-crops distort tiny 32x32 images a lot, so they are left out.)
train_aug_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4, padding_mode='reflect'),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])
train_aug_set = datasets.CIFAR10(root='./data', train=True, download=True, transform=train_aug_transform)
train_aug_loader = DataLoader(train_aug_set, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)

# Define Parameters
num_epochs = 10
lr = 3e-4
torch.manual_seed(0)
resnet18_aug = make_resnet18().to(device)

# Define Loss func and Optimizer
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(resnet18_aug.parameters(), lr=lr)

# Train Model
hist_aug = train(resnet18_aug, loss_fn, optimizer, train_aug_loader, test_loader, num_epochs, device)

# Plot Learning Curves
plot_learning_curve(hist_aug, 'ResNet18 (with augmentation)')

# Evaluate on Test Set
acc_aug, preds_aug, labels_aug = test_accuracy(resnet18_aug, test_loader, device)

### **3.2.3 Confusion Matrix**

Write code to plot a confusion matrix after training the above ResNet18. Please make sure the axis have class names (defined in Section 1)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

def plot_confusion_matrix(y_true, y_pred, classes, title='Confusion Matrix', normalize=True):
    cm = confusion_matrix(y_true, y_pred)
    if normalize:
        cm_disp = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    else:
        cm_disp = cm
    fig, ax = plt.subplots(figsize=(9, 7.5))
    im = ax.imshow(cm_disp, cmap='Blues')
    plt.colorbar(im, ax=ax, fraction=0.046)
    ax.set_xticks(range(len(classes))); ax.set_xticklabels(classes, rotation=45, ha='right')
    ax.set_yticks(range(len(classes))); ax.set_yticklabels(classes)
    ax.set_xlabel('Predicted label'); ax.set_ylabel('True label'); ax.set_title(title)
    thresh = cm_disp.max() / 2
    for i in range(len(classes)):
        for j in range(len(classes)):
            txt = f'{cm_disp[i, j]:.2f}' if normalize else f'{cm_disp[i, j]}'
            ax.text(j, i, txt, ha='center', va='center', fontsize=8,
                    color='white' if cm_disp[i, j] > thresh else 'black')
    plt.tight_layout()
    plt.show()

plot_confusion_matrix(labels_noaug, preds_noaug, classes, 'ResNet18 - no augmentation')
plot_confusion_matrix(labels_aug, preds_aug, classes, 'ResNet18 - with augmentation')

## **3.2.4 Freeze Network**

Initialize a new ResNet18, Freeze all the non-linear or non-fc weights. Hence, we will only be training the fc layers.

Freezing implies setting `requires_grad = False` for selected parameters


In [ ]:
# Instantiate Pretrained ResNet18
resnet18_frz = make_resnet18()

# Freeze all layers (except classifier/linear layers)
for name, param in resnet18_frz.named_parameters():
    param.requires_grad = name.startswith('fc.')

resnet18_frz = resnet18_frz.to(device)

# View Model to validate
print(resnet18_frz)
count_trainable(resnet18_frz)
print('Trainable tensors:', [n for n, p in resnet18_frz.named_parameters() if p.requires_grad])

### **3.2.5 Train frozen ResNet18**

Retrain the frozen ResNet18 now with the same setup as the previous ResNet18 (same batch size, optimizer, learning rate, etc.) for direct comparison. The results may not be as good that is expected.

In [ ]:
## TRAINING WITH AUGMENTATIONS (same setup as the best full fine-tune for a direct comparison)

# Define Parameters
num_epochs = 10
lr = 3e-4
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, resnet18_frz.parameters()), lr=lr)

# Train the model
hist_frz = train(resnet18_frz, loss_fn, optimizer, train_aug_loader, test_loader, num_epochs, device)

# Plot the learning curve
plot_learning_curve(hist_frz, 'ResNet18 frozen (fc only)')

# Evaluate on Test Set
acc_frz, preds_frz, labels_frz = test_accuracy(resnet18_frz, test_loader, device)
plot_confusion_matrix(labels_frz, preds_frz, classes, 'ResNet18 - frozen backbone (fc only)')

### ❓ **Q 3.2.6 What was the difference when training with and without data augmentations?**

**Answer:** (Compare the two learning-curve plots and accuracies printed above.)

- **Without augmentation** the full ResNet18 memorises the training set very quickly: training accuracy climbs to ~98–99% within a few epochs and training loss approaches 0. Validation accuracy plateaus early (roughly the low-to-mid 80s for 32×32 inputs), and the **validation loss stops decreasing and starts rising** after a few epochs. That widening train/validation gap is classic **overfitting**.
- **With augmentation** training accuracy rises more slowly and stays lower, because the task is harder: every epoch shows new random versions of the images. Validation accuracy, however, ends **higher** (typically a few percentage points), and validation loss keeps falling or stays flat instead of rising. The two curves stay much closer together, so the model **generalises better**.
- Per-class accuracies and the confusion matrix also improve, especially for classes that look similar (cat/dog, automobile/truck, deer/horse), because the model has learned features that are robust to position, flips and lighting instead of memorising specific images.
- Augmentation costs a little extra time per epoch (CPU work in the data loader) and would keep improving with more epochs, while the non-augmented model has already saturated.

___
### ❓ **Q 3.2.7 What is transfer learning and why would it be useful?**

**Answer:** Transfer learning means reusing a model trained on one (usually large) task as the starting point for a different target task. Here, ResNet18 was pretrained on ImageNet (1.28M images, 1000 classes). We keep its learned weights, replace the 1000-way output layer with a 10-way layer, and then either fine-tune the whole network or train only some of its layers on CIFAR10.

Why it is useful:
- **Reusable features:** the early layers of a CNN learn generic features (edges, colours, textures) and the middle layers learn parts and shapes. These are useful for almost any image task, so we don't have to learn them from scratch.
- **Less data needed:** a good starting point means we can reach high accuracy with far fewer labelled images and less risk of overfitting. This matters most for small datasets (medical imaging, Stanford Cars, etc.).
- **Faster training and less compute:** the model starts near a good solution, so it converges in a few epochs instead of hundreds.
- **Better final performance:** pretrained initialisations usually beat random initialisation, especially when the target dataset is small or similar to the source dataset.

___
### ❓ **Q 3.2.8 What was the difference between freezing layers and retraining all? Explain in terms of classification metrics and why you saw such metrics and confusuon matrix. Is what you say as expected?**

**Answer:**

**Metrics.** Retraining (fine-tuning) the whole network gives much higher accuracy than training only the fc layer. Full fine-tuning reaches roughly the high 80s on CIFAR10 at 32×32, while the frozen model typically ends around 40% with these settings. The frozen model's training and validation accuracy are both low and close together, a clear case of **underfitting**: its training loss plateaus high instead of approaching 0. In the frozen model's **confusion matrix** the diagonal is much weaker and errors are spread widely. It struggles most on the animal classes (cat, dog, deer, bird, frog), which get confused with each other, while vehicles (car, truck, ship, plane) are relatively better because they have more distinctive shapes and colours. The fully trained model's matrix is strongly diagonal, with remaining errors mostly in the classic cat↔dog and car↔truck pairs.

**Why.** With a frozen backbone only 5,130 parameters (512×10 + 10) are learned: a single linear classifier on fixed ImageNet features. Those features are a poor fit here:
1. **Resolution mismatch:** ResNet18 was trained on 224×224 images. On 32×32 CIFAR images the initial 7×7 stride-2 conv and max-pool shrink the input to 1×1 by the last layer, so the frozen features are coarse and don't represent these small images well.
2. **Domain shift:** the low-resolution CIFAR images have different statistics, and we normalise with (0.5, 0.5) rather than ImageNet's mean/std, so the frozen BatchNorm statistics and filters don't match the inputs.
3. **Capacity:** a linear layer can only separate classes that are already linearly separable in the frozen feature space.

Full fine-tuning adapts every layer to the new resolution and domain, so it performs far better.

**Is it expected?** Yes. The notebook itself anticipates that fc-only training will be worse. Linear probing works well when the target data closely resembles the pretraining data (same resolution and domain). Here it doesn't, so fine-tuning deeper layers is necessary. The gap would shrink if we resized CIFAR images to 224×224 and used ImageNet normalisation.

___
### ❓ **Q 3.2.9 For a pretrained model like ResNet, in what general scenarios would you (1) freeze your pretrained weights adn retrain last layer, (2) retrain entire architecture, and (3) Freeze all convolution layer weights and retrain all FC layers? Explain with example scenarios if possible.**

**Answer:** The choice depends on two things: **how much target data we have** and **how similar it is to the pretraining data**.

1. **Freeze the pretrained weights and retrain only the last layer (linear probe).**
   *When:* the target dataset is **small** and **similar** to the source domain. The pretrained features already describe the data well, and training more parameters on little data would overfit.
   *Example:* classifying 5 dog breeds with a few hundred photos using an ImageNet model; recognising a handful of product types from ordinary photos; quick baselines or very limited compute (e.g. on-device training).

2. **Retrain (fine-tune) the entire architecture.**
   *When:* the target dataset is **large**, or **quite different** from the source domain, so all layers, including low-level filters, need to adapt. There is enough data to do so without overfitting.
   *Example:* CIFAR10 at 32×32 (as we saw), medical X-rays/CT scans, satellite imagery, or fine-grained datasets like Stanford Cars with thousands of images. Use a smaller learning rate than training from scratch so the pretrained knowledge isn't destroyed.

3. **Freeze all convolutional layers and retrain all the FC layers (classifier head).**
   *When:* there is a **moderate** amount of data that is reasonably **similar** to the source. The convolutional features are good, but a more powerful, non-linear classifier is needed than a single linear layer: for example, when the head has several FC layers (VGG/AlexNet-style) or a new multi-layer head is attached for a new task.
   *Example:* using a VGG16 backbone for a 50-class flower or food dataset with a few thousand images; adding a 2–3-layer MLP head for multi-label tagging of everyday photos.

A common middle ground (used in 3.2.10) is to freeze the early layers, which hold generic edges and textures, and fine-tune the later, more task-specific blocks together with the head.

___


### **3.2.10 Train frozen ResNet18 for better performance**

You may have noticed only training just the fc layer did not yield good performance. Now, you must try to investigate trying to freeze as much of the original model as possible for best results.


*   Keep fc layer unfrozen, and unfreeze the last block (in layer4 of resnet18)
*   You pick which layers to unfreeze!

Investigate the maximize the layers to unfreeze to yield best balance between performance to training time.

In [ ]:
# Instantiate Pretrained ResNet18
resnet18_frz = make_resnet18()

# Unfreeze the last block (in layer4 of resnet18): freeze everything, then unfreeze layer4[-1] + fc
for param in resnet18_frz.parameters():
    param.requires_grad = False
for param in resnet18_frz.layer4[-1].parameters():
    param.requires_grad = True
for param in resnet18_frz.fc.parameters():
    param.requires_grad = True
resnet18_frz = resnet18_frz.to(device)
count_trainable(resnet18_frz)

## TRAINING WITH AUGMENTATIONS
# Define Parameters
num_epochs = 10
lr = 3e-4
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, resnet18_frz.parameters()), lr=lr)

# Train the model
hist_l4 = train(resnet18_frz, loss_fn, optimizer, train_aug_loader, test_loader, num_epochs, device)

# Plot the learning curve
plot_learning_curve(hist_l4, 'ResNet18: last block of layer4 + fc unfrozen')

# Evaluate on Test Set
acc_l4, _, _ = test_accuracy(resnet18_frz, test_loader, device)


In [ ]:
# Instantiate Pretrained ResNet18
resnet18_frz = make_resnet18()

# Freeze whichever layers you choose for best performance
# Choice: freeze the stem + layer1 + layer2 (generic edges/textures), train layer3 + layer4 + fc.
for param in resnet18_frz.parameters():
    param.requires_grad = False
for block in [resnet18_frz.layer3, resnet18_frz.layer4, resnet18_frz.fc]:
    for param in block.parameters():
        param.requires_grad = True
resnet18_frz = resnet18_frz.to(device)
count_trainable(resnet18_frz)

## TRAINING WITH AUGMENTATIONS
# Define Parameters
num_epochs = 10
lr = 3e-4
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, resnet18_frz.parameters()), lr=lr)

# Train the model
hist_l34 = train(resnet18_frz, loss_fn, optimizer, train_aug_loader, test_loader, num_epochs, device)

# Plot the learning curve
plot_learning_curve(hist_l34, 'ResNet18: layer3 + layer4 + fc unfrozen')

# Evaluate on Test Set
acc_l34, _, _ = test_accuracy(resnet18_frz, test_loader, device)

# Summary: accuracy vs training time for every configuration
summary = [('fc only', hist_frz), ('layer4[-1] + fc', hist_l4),
           ('layer3 + layer4 + fc', hist_l34), ('full fine-tune', hist_aug)]
print(f"\n{'Configuration':<24}{'Best val acc':>14}{'Total time (s)':>16}")
for name, h in summary:
    print(f"{name:<24}{max(h['val_acc']):>13.2f}%{sum(h['epoch_time']):>16.1f}")



### ❓ **Q 3.2.11 Which layers did you freeze to yield the best performance vs training time balance? Can you give any intuition as to why this worked well? Compare it with unfreezing only the fc layer and explain why it performs better.**

**Answer:** (Use the summary table printed above for exact numbers.)

**Best balance:** freezing the **stem, layer1 and layer2** and training **layer3 + layer4 + fc** (~94% of the parameters, but only part of the compute). Its accuracy comes close to full fine-tuning, while each epoch is faster: no gradients are computed or stored for the early, high-resolution layers, which hold most of the activations, so the backward pass is cheaper and uses less memory. Unfreezing only the last block of layer4 is faster still and already a large jump over fc-only, but it remains a few points behind because layer3 can't adapt.

**Intuition:** CNN layers go from generic to task-specific. The stem, layer1 and layer2 learn edges, colour blobs and simple textures, which transfer well to almost any natural image, including CIFAR. Freezing them loses little and also acts as a regulariser. The deeper layers (layer3/layer4) encode ImageNet-specific object parts and work at the lowest spatial resolution, which is exactly where the 32×32 CIFAR images differ most from 224×224 ImageNet images. Re-training them lets the network rebuild high-level features for the new resolution and classes.

**Compared with fc-only:** fc-only learns a single linear map on frozen, mismatched features, so it underfits (~40%). Training even one residual block of layer4 lets the model reshape the feature space so the 10 classes become separable. Once the deeper blocks are unfrozen, the model has enough adaptable capacity to fit CIFAR10, and accuracy roughly doubles, while freezing the early layers keeps training cheaper than fine-tuning everything.

___
___
# End of Phase 2 😊🥳